# 04_llm_analysis

Use the **OpenAI API** (`gpt-3.5-turbo`) to automatically generate concise scouting
reports for the top Premier League players based on their statistics.
Reports are stored back in the SQLite database for display in the Flask dashboard.

## Setup & Imports

This notebook uses the **OpenAI API** (`gpt-3.5-turbo`) — an API key is required.

**One-time setup:**

1. Create an API key at <https://platform.openai.com/api-keys>.
2. Add it to the `.env` file in the project root:
   ```
   OPENAI_API_KEY=your_key_here
   ```
3. Make sure your account has available credits at <https://platform.openai.com/usage>.

In [1]:
# Standard-library and third-party imports
import os
import time
import sqlite3

import pandas as pd
import openai
from dotenv import load_dotenv

# ── OpenAI client setup ────────────────────────────────────────────────────────
load_dotenv()

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ── Load cleaned player data from SQLite ──────────────────────────────────────
os.makedirs("data", exist_ok=True)
db_path = "data/football.db"
connection = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM players_cleaned", connection)
connection.close()

print(f"✓ Loaded {len(df)} players from players_cleaned")
print(f"Columns: {list(df.columns)}")
display(df.head(3))

✓ Loaded 351 players from players_cleaned
Columns: ['player_id', 'name', 'nationality', 'date_of_birth', 'team_id', 'team_name', 'position', 'market_value', 'market_value_numeric', 'market_value_tm', 'position_group', 'age', 'has_market_value']


,player_id,name,nationality,date_of_birth,team_id,team_name,position,market_value,market_value_numeric,market_value_tm,position_group,age,has_market_value
0,1731,Gianluigi Donnarumma,Italy,1999-02-25,65,Manchester City FC,Goalkeeper,14.2,14.2,14.2,Goalkeeper,27,1
1,3953,Marcus Bettinelli,England,1992-05-24,65,Manchester City FC,Goalkeeper,6.4,6.4,6.4,Goalkeeper,34,1
2,153874,James Trafford,England,2002-10-10,65,Manchester City FC,Goalkeeper,5.8,5.8,5.8,Goalkeeper,23,1


## LLM Helper Function

`generate_scouting_report(player)` takes a single player row and calls
**gpt-3.5-turbo** via the OpenAI API to produce a concise, 3-4 sentence scouting report.

In [2]:
def generate_scouting_report(player: pd.Series) -> str:
    """
    Generate a 3-4 sentence scouting report for a single player using
    gpt-3.5-turbo via the OpenAI API.

    Args:
        player (pd.Series): A single row from the players_cleaned DataFrame.

    Returns:
        str: Scouting report text, or a descriptive error message.
    """
    # Extract relevant statistics; fall back gracefully on missing values
    name         = player.get("name",           "Unknown Player")
    position     = player.get("position_group", "Unknown")
    nationality  = player.get("nationality",    "Unknown")
    age          = player.get("age",            "N/A")
    market_value = player.get("market_value_tm", None)

    # Format the market value for readability
    if market_value is not None and not pd.isna(market_value):
        mv_str = f"€{float(market_value):.1f}M"
    else:
        mv_str = "not available"

    # Build the prompt sent to the model
    prompt = (
        f"You are a professional football scout. Write a concise 3-4 sentence "
        f"scouting report for the following player based solely on the provided stats:\n\n"
        f"- Name: {name}\n"
        f"- Position: {position}\n"
        f"- Nationality: {nationality}\n"
        f"- Age: {age}\n"
        f"- Estimated Market Value: {mv_str}\n\n"
        f"Focus on potential, playing style expectations for their position, "
        f"and transfer market outlook. Be professional and factual."
    )

    # Call the OpenAI API with error handling
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an expert football scout who writes clear, "
                        "professional scouting reports in 3-4 sentences."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            max_tokens=200,
            temperature=0.7,
        )
        # Extract the generated text from the response
        return response.choices[0].message.content.strip()

    except openai.AuthenticationError:
        return "Error: Invalid or missing OPENAI_API_KEY. Check your .env file."
    except openai.RateLimitError:
        return "Error: OpenAI rate limit or quota exceeded. Check your usage at platform.openai.com/usage."
    except openai.APIConnectionError:
        return "Error: Cannot connect to the OpenAI API. Check your internet connection."
    except openai.APIStatusError as e:
        return f"OpenAI API error {e.status_code}: {e.message}"
    except Exception as e:
        return f"Unexpected error: {e}"

print("✓ generate_scouting_report() defined — using gpt-3.5-turbo via OpenAI API")

✓ generate_scouting_report() defined — using gpt-3.5-turbo via OpenAI API


## Generate Reports

Select the top 5 players by market value and generate a scouting report for each.
A 1-second pause is added between calls to stay within OpenAI rate limits.

In [3]:
# Select the top 5 players by market value (descending)
top5 = (
    df.dropna(subset=["market_value_tm"])
    .sort_values("market_value_tm", ascending=False)
    .head(5)
    .copy()
    .reset_index(drop=True)
)

print(f"Generating scouting reports for {len(top5)} players...\n")

# Store generated reports in a new column
reports = []

for idx, player in top5.iterrows():
    name = player.get("name", f"Player {idx}")
    print(f"[{idx + 1}/5] Generating report for {name}...")

    # Call the LLM helper function
    report = generate_scouting_report(player)
    reports.append(report)

    # Print the generated scouting report
    print(f"\n  Report for {name}:")
    print(f"  {report}\n")

    # Wait 1 second between requests to stay within rate limits
    if idx < len(top5) - 1:
        time.sleep(1)

# Add the reports as a new column on the top-5 DataFrame
top5["scouting_report"] = reports

print("✓ All scouting reports generated")
display(top5[["name", "position_group", "market_value_tm", "scouting_report"]])

Generating scouting reports for 5 players...

[1/5] Generating report for Zach Marsh...

  Report for Zach Marsh:
  Zach Marsh, a 20-year-old English forward, possesses an impressive estimated market value of €118.0M, indicating high potential and market demand. His value suggests he is a rising talent with a bright future in the football industry. Given his young age and nationality, Marsh is likely to attract significant interest from top clubs looking to secure a promising forward for the long term.

[2/5] Generating report for Joshua Ajala...

  Report for Joshua Ajala:
  Joshua Ajala is a promising 19-year-old English forward with an estimated market value of €115.8M. He possesses exceptional speed and technical ability, making him a dynamic threat in the attacking third. Given his age and potential, Ajala is likely to attract significant interest from top clubs looking to secure a long-term solution in their forward line.

[3/5] Generating report for Keiber Lamadrid...

  Report 

,name,position_group,market_value_tm,scouting_report
0,Zach Marsh,Forward,118.0,"Zach Marsh, a 20-year-old English forward, pos..."
1,Joshua Ajala,Forward,115.8,Joshua Ajala is a promising 19-year-old Englis...
2,Keiber Lamadrid,Forward,115.2,"Keiber Lamadrid, a 22-year-old Venezuelan forw..."
3,Sean Neave,Forward,108.9,"Sean Neave, an 18-year-old English forward, ha..."
4,James Wilson,Forward,99.4,James Wilson is a promising 19-year-old Scotti...


## Save to Database

Persist the top-5 players together with their scouting reports to SQLite so the Flask dashboard can load and display them without re-calling the API.

In [4]:
# Save top-5 players + scouting reports to the database
os.makedirs("data", exist_ok=True)
connection = sqlite3.connect(db_path)

try:
    # Replace the table on every run so reports stay up-to-date
    top5.to_sql("player_reports", connection, if_exists="replace", index=False)

    saved = pd.read_sql_query(
        "SELECT name, position_group, scouting_report FROM player_reports", connection
    )
    print(f"✓ Saved {len(saved)} records to player_reports table")
    print("\nVerification:")
    display(saved)

except Exception as e:
    print(f"✗ Error saving to database: {e}")

finally:
    # Always close the database connection
    connection.close()
    print("✓ Database connection closed")

✓ Saved 5 records to player_reports table

Verification:


,name,position_group,scouting_report
0,Zach Marsh,Forward,"Zach Marsh, a 20-year-old English forward, pos..."
1,Joshua Ajala,Forward,Joshua Ajala is a promising 19-year-old Englis...
2,Keiber Lamadrid,Forward,"Keiber Lamadrid, a 22-year-old Venezuelan forw..."
3,Sean Neave,Forward,"Sean Neave, an 18-year-old English forward, ha..."
4,James Wilson,Forward,James Wilson is a promising 19-year-old Scotti...


✓ Database connection closed
